<a href="https://colab.research.google.com/github/GustavoCastillo1997/anime_recomendation/blob/main/NLP_Project_Sistema_de_Recomenda%C3%A7%C3%A3o_de_Animes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download do dataset


import kagglehub


print('Baixando dataset...')
path = kagglehub.dataset_download("svanoo/myanimelist-dataset")

print('Caminho até o dataset:', path)


Baixando dataset...
Caminho até o dataset: /kaggle/input/myanimelist-dataset


In [ ]:
# Import das bibliotecas necessárias


import string
import nltk
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Input de dados


path = path + '/anime.csv'

df_anime = pd.read_csv(path, sep='\t')


In [ ]:
# Filtrando colunas de interesse


df_anime = df_anime.iloc[:, [2, 3, 12, 13]]


In [ ]:
# Inicializando dicionário e lista de sinopses para tratamento dos dados


anime_base = {}


for title in df_anime['title']:
    anime_base[title] = None


synopsis_list = [synopsis for synopsis in df_anime['synopsis']]


In [ ]:
# Criando listas de tokens


tokenized_lists = []


for index, synopsis in enumerate(synopsis_list):
  if isinstance(synopsis, float):
      synopsis_list[index] = ''


for index, synopsis in enumerate(synopsis_list):
        tokenized_lists.append(nltk.tokenize.word_tokenize(synopsis))


In [ ]:
# Remoção de Stopwords


tokenized_list_clean_1 = []


for list_of_tokens in tokenized_lists:
    tokenized_sublist = []
    for token in list_of_tokens:
        if not token.lower() in nltk.corpus.stopwords.words('english') and not token in string.punctuation:
            tokenized_sublist.append(token)
    tokenized_list_clean_1.append(tokenized_sublist)


In [ ]:
# Remoção de mais alguns ruídos (listas vazias, reticiências, aspas simples)


tokenized_list_clean_2 = []

remanescent_data_noise = ['...', "''", '``', '-', '’', "'s", "n't", "'ll"]


for list_of_tokens in tokenized_list_clean_1:
    tokenized_sublist = []
    for token in list_of_tokens:
        if not token in remanescent_data_noise:
            tokenized_sublist.append(token)
    tokenized_list_clean_2.append(tokenized_sublist)


In [ ]:
# Tratamento de dados da coluna GÊNERO


genre_list = []


for index, genre in enumerate(df_anime['genres']):
    if isinstance(genre, str):
        genre_list.append(genre)
        if '|' in genre_list[index]:
            genre_list[index] = genre_list[index].split('|')
    else:
        genre_list.append('')


In [ ]:
# Tratamento de dados da coluna ESTÚDIO


studio_list = []


for index, genre in enumerate(df_anime['studios']):
    if isinstance(genre, str):
        studio_list.append(genre)
        if '|' in studio_list[index]:
            studio_list[index] = studio_list[index].split('|')
    else:
        studio_list.append('')


In [ ]:
# Agrupando todas as informações em uma única lista


for index, synopsis in enumerate(tokenized_list_clean_2):
    if isinstance(genre_list[index], list):
        for genre in genre_list[index]:
            tokenized_list_clean_2[index].append(genre)
    else:
        tokenized_list_clean_2[index].append(genre_list[index])
    if isinstance(studio_list[index], list):
        for studio in studio_list[index]:
            tokenized_list_clean_2[index].append(studio)
    elif studio_list[index] == '':
        continue
    else:
        tokenized_list_clean_2[index].append(studio_list[index])


In [ ]:
# Inserindo as listas de tokens no dicionário de animes

index = 0

for anime in anime_base:
    anime_base[anime] = tokenized_list_clean_2[index]
    index += 1


In [ ]:
# Entrada de dados


user_input = []
token = input('Insira as palavras-chave. Digite "0" para finalizar:\n')

while token != '0':
    user_input.append(token)
    token = input()

Insira as palavras-chave. Digite "0" para finalizar:
naruto
0


In [ ]:
# Vetorização e Verificação de Similaridade


anime_names = list(anime_base.keys())
docs = [" ".join(anime_base[name]) for name in anime_names]
input_str = " ".join(user_input)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs + [input_str])

cos_sim = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])[0]
top_matches = np.argsort(cos_sim)[::-1][:3]

In [ ]:
# Exibição dos resultados


print("Top recomendações:")
for position, index in enumerate(top_matches):
    name = anime_names[index]
    print(f'{position+1}. {name} - Similaridade: {cos_sim[index]:.5f}')

Top recomendações:
1. Juliet - Similaridade: 0.46135
2. Naruto: Shippuuden Movie 6 - Road to Ninja - Similaridade: 0.41992
3. Naruto - Similaridade: 0.41914
